## Stage 1.4.2.5.1 — Basic OCR
Objective

In this stage, we will take the scanned PDF we already have, convert one PDF page into an image, run Tesseract OCR against that image, and obtain text.

    Scanned PDF
        ↓
    PDF Page
        ↓
    Page Image
        ↓
    Tesseract OCR
        ↓
    Extracted Text
        ↓
    Save OCR Text

### Step 1 — Understand the Basic OCR Flow

A scanned PDF does not necessarily contain an actual text layer.

Therefore, we cannot simply extract text from it like a normal text-based PDF.

Instead:

     Scanned PDF
          ↓
     Render PDF page
          ↓
     Image
          ↓
     OCR
          ↓
     Text

Tesseract fundamentally works with images, not directly with the visual appearance of a PDF page.

Therefore, our first task is to convert the PDF page into an image.

### Step 2 — Understand the Two OCR Components

There are two separate components involved.

Component 1 — Tesseract

Tesseract is the actual OCR engine.

Image → Tesseract → Text
Component 2 — pytesseract

pytesseract is the Python wrapper that allows Python to communicate with Tesseract.

      Python
         ↓
      pytesseract
         ↓
      Tesseract OCR
         ↓
      Image → Text

Therefore:

      Windows
      └── Tesseract OCR Engine


      Python environment
      └── pytesseract

These are not the same thing.

### Step 3 — Install Tesseract OCR on Windows

Install the Windows version of Tesseract.

Use the UB Mannheim Windows distribution:

Tesseract Windows installation information

During installation, make sure the English (eng) language data is installed.

A common installation location is:

C:/Program Files/Tesseract-OCR/tesseract.exe

### Step 4 — Verify Tesseract from PowerShell

Open a new PowerShell terminal.

Run:

tesseract --version

You should get something similar to:

tesseract 5.x.x
leptonica-...

Now check the installed languages:

tesseract --list-langs

You should see at least:

eng
osd
Why are we checking this?

We want to verify that the OCR engine itself is correctly installed before involving Python.

### Step 5 — Install pytesseract Using UV

Go to the root of your existing rag-learning project.

Run:

uv add pytesseract

Remember:

    uv
    │
    └── manages Python dependencies
            │
            └── pytesseract

Whereas:

    Windows
    │
    └── Tesseract OCR executable

So uv installs the Python wrapper; it does not install the Tesseract engine itself.

### Step 6 — Verify Tesseract from the Notebook

Open your Stage 1.4.2 notebook.

Run:

In [ ]:
import pytesseract

print(pytesseract.get_tesseract_version())

If successful, you should see the installed Tesseract version.

Now check the available languages:

In [ ]:
print(pytesseract.get_languages())

You should see eng in the result.

### Step 7 — Handle TesseractNotFoundError if Necessary

If Python gives you something like:

TesseractNotFoundError

it usually means:

Python
   ↓
pytesseract
   ↓
❌ Cannot locate tesseract.exe

In that situation, explicitly configure the executable path:

In [ ]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:/Program Files/Tesseract-OCR/tesseract.exe"
)

Then test again:

In [ ]:
print(pytesseract.get_tesseract_version())

If it works, Python can now communicate with Tesseract.

### Step 8 — Locate Our Scanned PDF

Now we come to an important correction from our previous discussion.

We do not already have scanned_page.png.

We have the scanned PDF.

Therefore, we will create scanned_page.png ourselves.

First specify the actual path of your scanned PDF:

In [2]:
from pathlib import Path

pdf_path = Path("D:/AI Learning/rag-learning/notebooks/PDF-loaders/extracted_images/page_1_img_1.png")

Check that the file exists:

In [3]:
print(pdf_path.exists())
print(pdf_path)

False
D:\AI Learning\rag-learning\notebooks\PDF-loaders\extracted_images\page_1_img_1.png


We want:

True

### Step 9 — Open the Scanned PDF

We will use PyMuPDF, which you already learned about during the PDF-loader comparison.

Import it:

In [4]:
import fitz

Open the PDF:

In [5]:
pdf = fitz.open(pdf_path)

FileNotFoundError: no such file: 'D:\AI Learning\rag-learning\notebooks\PDF-loaders\extracted_images\page_1_img_1.png'

Check the number of pages:

In [ ]:
print("Number of pages:", len(pdf))

### Step 10 — Select the First PDF Page

For our first OCR experiment, we will deliberately process only one page.

Select page 1:

In [ ]:
page = pdf[0]

Remember that Python uses zero-based indexing:

    pdf[0] → Page 1
    pdf[1] → Page 2
    pdf[2] → Page 3

Step 11 — Render the PDF Page as an Image

Now convert the PDF page into an image representation.

In [ ]:
pix = page.get_pixmap()

Conceptually:

     Scanned PDF
          ↓
     PyMuPDF
          ↓
     PDF Page
          ↓
     Pixmap

Step 12 — Save the Page as scanned_page.png

Now save the rendered page:

In [ ]:
image_path = "scanned_page.png"

pix.save(image_path)

print(f"Saved: {image_path}")

Now we have actually created:

    scanned_page.png

This file did not come with the original exercise.

We generated it from the scanned PDF.

### Step 13 — Load the Generated Image

Now use Pillow to open the generated image:

In [ ]:
from PIL import Image

image = Image.open(image_path)

Check its properties:

In [ ]:
print("Image size:", image.size)
print("Image mode:", image.mode)

### Step 14 — Display the Page Image

Before running OCR, visually inspect the image.

In [ ]:
from IPython.display import display

display(image)

At this point we should be able to see the scanned page that Tesseract will process.

This is important because OCR doesn't see the PDF the way we do.

We see:

Text
Tables
Headings
Images
Diagrams

The OCR engine receives:

Pixels

### Step 15 — Run Basic OCR

Now we perform the actual OCR.

In [ ]:
ocr_text = pytesseract.image_to_string(
    image,
    lang="eng"
)

Print the result:

In [ ]:
print(ocr_text)

Our pipeline has now become:

     Scanned PDF
          ↓
     PDF Page
          ↓
     scanned_page.png
          ↓
     Tesseract
          ↓
     OCR text

🎉 This is our first Basic OCR implementation.

### Step 16 — Inspect the OCR Result

Let's measure what was extracted.

In [ ]:
print("Characters extracted:", len(ocr_text))

Count the lines:

In [ ]:
print("Lines extracted:", len(ocr_text.splitlines()))

And display the complete result:

In [ ]:
print("=" * 80)
print(ocr_text)
print("=" * 80)

Now compare the OCR output with the original scanned page.

Check:

Are the words correct?
Are numbers correct?
Are headings recognized?
Are paragraphs preserved?
Are special characters correct?
Are lines in the correct order?
Did anything disappear?

We will use these observations later for OCR Quality Evaluation.

### Step 17 — Save the OCR Result

Save the extracted text:

In [29]:
ocr_output_path = "scanned_page_ocr.txt"

with open(
    ocr_output_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(ocr_text)

print(f"OCR output saved to: {ocr_output_path}")

OCR output saved to: scanned_page_ocr.txt


Now we have:

     Scanned PDF
          │
          ├── scanned_page.png
          │
          └── scanned_page_ocr.txt

### Step 18 — Understand What We Have Achieved

We started with:

Scanned PDF

and ended with:

OCR Text

The complete process is:

        Step 1
        Scanned PDF
            ↓
        Step 2
        PDF Page
            ↓
        Step 3
        Page → Image
            ↓
        Step 4
        Image → Tesseract
            ↓
        Step 5
        OCR → Text

This is the fundamental OCR workflow.

### Step 19 — Understand Where OCR Fits into RAG

OCR itself is not RAG.

OCR is part of document ingestion.

Our larger pipeline is:

                 RAG INGESTION
                      │
                      ▼
                  PDF Input
                      │
            ┌─────────┴─────────┐
            │                   │
       Text PDF            Scanned PDF
            │                   │
            ▼                   ▼
      Text Extraction       PDF → Image
                                │
                                ▼
                              OCR
                                │
                                ▼
                         Extracted Text
            │                   │
            └─────────┬─────────┘
                      ▼
                LangChain Docs
                      │
                      ▼
                   Chunking
                      │
                      ▼
                  Embeddings
                      │
                      ▼
                  Vector DB
                      │
                      ▼
                  Retrieval
                      │
                      ▼
                     LLM

Therefore, OCR allows scanned documents to enter our text-based RAG pipeline.

### Step 20 — Understand the Limitation of Basic OCR

Basic OCR does not necessarily understand document structure.

Suppose the scanned PDF contains:

Name       Department       Experience
Ravi       Data             8
Suresh     IT               10

Tesseract might extract something like:

Name Department Experience
Ravi Data 8
Suresh IT 10

The words may be correct, but the table structure may be lost.

The same issue can occur with:

    Tables
    Multiple columns
    Charts
    Architecture diagrams
    Mathematical formulas
    Headers and footers
    Images
    Complex layouts

This is exactly why our OCR learning journey continues beyond Basic OCR.

### Step 21 — Understand Our Next Stages

Our OCR learning structure is:

        Stage 1.4.2.5 — OCR Approach
                │
                ├── Stage 1.4.2.5.1 — Basic OCR       ← CURRENT
                │
                ├── Stage 1.4.2.5.2 — OCR Preprocessing
                │
                ├── Stage 1.4.2.5.3 — OCR Quality Evaluation
                │
                ├── Stage 1.4.2.5.4 — OCR + PDF Page Pipeline
                │
                └── Stage 1.4.2.5.5 — OCR → LangChain Documents

We should not jump to preprocessing yet.

First, we want to run this basic OCR against your actual scanned PDF and examine the result.